# House Prices: Data Preprocessing for ML

This notebook focuses on cleaning, preprocessing, and preparing data for machine learning models.

**Key Tasks:**
1. Handle missing values properly
2. Encode categorical and ordinal features
3. Scale numerical features
4. Create feature engineering pipeline
5. Prepare train/test splits for modeling

In [26]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

from src.utils import (
    load_data, identify_na_as_absence_columns, replace_na_with_none,
    numeric_vs_categorical_split
)

In [27]:
# Load raw data
train, test, sample = load_data("../data/")

print("Original training data shape:", train.shape)
print("Original test data shape:", test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)
Sample submission shape: (1459, 2)
Original training data shape: (1460, 81)
Original test data shape: (1459, 80)


## 1. Handle "Absence" NA Values

In [28]:
# Columns where NA means "absence of feature"
absence_cols = identify_na_as_absence_columns()
print(f"Columns to convert NA to 'None':\n{absence_cols}\n")

# Replace NA with 'None'
train_processed = replace_na_with_none(train, absence_cols)
test_processed = replace_na_with_none(test, absence_cols)

print("After processing:")
print(f"Training set - NA count: {train_processed.isnull().sum().sum()}")
print(f"Test set - NA count: {test_processed.isnull().sum().sum()}")

Columns to convert NA to 'None':
['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']

After processing:
Training set - NA count: 1221
Test set - NA count: 1236


## 2. Define Ordinal Feature Mappings

In [29]:
# Ordinal features and their mappings (Ex=5, Gd=4, TA=3, Fa=2, Po=1, None=0)
ordinal_features = {
    'ExterQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'ExterCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'BsmtQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'BsmtCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'BsmtExposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0},
    'BsmtFinType1': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0},
    'BsmtFinType2': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0},
    'HeatingQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'KitchenQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'FireplaceQu': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'GarageQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'GarageCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'PoolQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'Fence': {'GdPrv': 4, 'MnPrv': 3, 'GdWo': 2, 'MnWw': 1, 'None': 0}
}

# Apply ordinal encoding
for feature, mapping in ordinal_features.items():
    if feature in train_processed.columns:
        train_processed[feature] = train_processed[feature].map(mapping).fillna(0)
        test_processed[feature] = test_processed[feature].map(mapping).fillna(0)

print("Ordinal encoding applied successfully")

Ordinal encoding applied successfully


## 3. Handle Remaining Missing Values

In [30]:
# Identify remaining missing values
train_missing = train_processed.isnull().sum()
train_missing = train_missing[train_missing > 0].sort_values(ascending=False)

test_missing = test_processed.isnull().sum()
test_missing = test_missing[test_missing > 0].sort_values(ascending=False)

print("Remaining missing values in training set:")
print(train_missing.to_string())
print("\nRemaining missing values in test set:")
print(test_missing.to_string())


Remaining missing values in training set:
MasVnrType     872
LotFrontage    259
GarageYrBlt     81
MasVnrArea       8
Electrical       1

Remaining missing values in test set:
MasVnrType      894
LotFrontage     227
GarageYrBlt      78
MasVnrArea       15
MSZoning          4
Utilities         2
BsmtHalfBath      2
Functional        2
BsmtFullBath      2
Exterior1st       1
BsmtUnfSF         1
BsmtFinSF2        1
BsmtFinSF1        1
Exterior2nd       1
TotalBsmtSF       1
GarageCars        1
GarageArea        1
SaleType          1


In [31]:
# Imputation strategy 
# Note (Bryce): This is the simplest imputation stategy. Obviously this is not perfect, and can alter the probability distribution of values in training.
#               This issue could be addressed in the future with conditional/reggressional/MI methods (Advance - have to refer to notes)
# Numerical: median imputation
# Categorical: most frequent imputation (mode)

numeric_cols, categorical_cols = numeric_vs_categorical_split(train_processed, exclude_cols=['Id', 'SalePrice'])

# Create imputers
numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

# Apply imputation
train_processed[numeric_cols] = numeric_imputer.fit_transform(train_processed[numeric_cols])
train_processed[categorical_cols] = categorical_imputer.fit_transform(train_processed[categorical_cols])

test_processed[numeric_cols] = numeric_imputer.transform(test_processed[numeric_cols])
test_processed[categorical_cols] = categorical_imputer.transform(test_processed[categorical_cols])

# Verify no missing values remain
print(f"Missing values in training after imputation: {train_processed.isnull().sum().sum()}")
print(f"Missing values in test after imputation: {test_processed.isnull().sum().sum()}")

Missing values in training after imputation: 0
Missing values in test after imputation: 0


## 4. Feature Engineering

In [32]:
def engineer_features(df):
    """Add engineered features to the dataframe."""
    df_eng = df.copy()
    
    # Total living area (above ground + basement)
    df_eng['TotalSF'] = df_eng['GrLivArea'] + df_eng['TotalBsmtSF']
    
    # Total bathrooms (full + half)
    # Half Bbathrooms treated with half the weighting
    df_eng['TotalBath'] = (df_eng['FullBath'] + 
                           0.5 * df_eng['HalfBath'] + 
                           df_eng['BsmtFullBath'] + 
                           0.5 * df_eng['BsmtHalfBath'])
    
    # Total porch area
    df_eng['TotalPorchSF'] = (df_eng['OpenPorchSF'] + 
                               df_eng['EnclosedPorch'] + 
                               df_eng['3SsnPorch'] + 
                               df_eng['ScreenPorch'])
    
    # Age features
    current_year = 2010  # Last year in dataset
    df_eng['HouseAge'] = current_year - df_eng['YearBuilt']
    df_eng['RemodAge'] = current_year - df_eng['YearRemodAdd']
    df_eng['IsRemodeled'] = (df_eng['YearBuilt'] != df_eng['YearRemodAdd']).astype(int)
    
    # Has basement/garage/pool/fireplace (binary)
    df_eng['HasBsmt'] = (df_eng['TotalBsmtSF'] > 0).astype(int)
    df_eng['HasGarage'] = (df_eng['GarageArea'] > 0).astype(int)
    df_eng['HasPool'] = (df_eng['PoolArea'] > 0).astype(int)
    df_eng['HasFireplace'] = (df_eng['Fireplaces'] > 0).astype(int)
    
    # Quality features (sum of ordinal qualities)
    quality_cols = ['ExterQual', 'BsmtQual', 'HeatingQC', 'KitchenQual', 'GarageQual']
    df_eng['TotalQual'] = df_eng[quality_cols].sum(axis=1)
    
    # Area per room
    df_eng['AreaPerRoom'] = df_eng['GrLivArea'] / df_eng['TotRmsAbvGrd']
    
    return df_eng

In [33]:
# Apply feature engineering
train_processed = engineer_features(train_processed)
test_processed = engineer_features(test_processed)

print("Feature engineering complete")
print(f"New training shape: {train_processed.shape}")
print(f"New test shape: {test_processed.shape}")

Feature engineering complete
New training shape: (1460, 93)
New test shape: (1459, 92)


In [34]:
# Update numeric/categorical split after engineering
numeric_cols, categorical_cols = numeric_vs_categorical_split(train_processed, exclude_cols=['Id', 'SalePrice'])
print(f"\nUpdated numeric features: {len(numeric_cols)}")
print(f"Updated categorical features: {len(categorical_cols)}")


Updated numeric features: 62
Updated categorical features: 29


## 5. Handle Categorical Features

In [35]:
# Identify high cardinality features for special handling
cardinality = {}
for col in categorical_cols:
    cardinality[col] = train_processed[col].nunique()

high_cardinality = [col for col, val in cardinality.items() if val > 10]
low_cardinality = [col for col, val in cardinality.items() if 2 <= val <= 10]

print(f"High cardinality features (>10 unique values): {high_cardinality}")
print(f"Low cardinality features (2-10 unique values): {low_cardinality[:10]}...")


High cardinality features (>10 unique values): ['Neighborhood', 'Exterior1st', 'Exterior2nd']
Low cardinality features (2-10 unique values): ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2']...


In [37]:
# Option 1: One-hot encoding for low cardinality features
# Option 2: Target encoding for high cardinality (will be done in modeling notebook)
# For now, we'll prepare both options

# One-hot encode low cardinality features
train_encoded = pd.get_dummies(train_processed, columns=low_cardinality, drop_first=True)
test_encoded = pd.get_dummies(test_processed, columns=low_cardinality, drop_first=True)

# Align columns (test might have missing categories)
train_cols = set(train_encoded.columns)
test_cols = set(test_encoded.columns)

# Add missing columns to test
for col in train_cols - test_cols:
    if col != 'SalePrice':
        test_encoded[col] = 0

# Remove extra columns from test
for col in test_cols - train_cols:
    if col != 'Id':
        test_encoded = test_encoded.drop(columns=[col])

print(f"Encoded training shape: {train_encoded.shape}")
print(f"Encoded test shape: {test_encoded.shape}")

Encoded training shape: (1460, 176)
Encoded test shape: (1459, 175)
